In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
seiftarek158_contract_nli_path = kagglehub.dataset_download('seiftarek158/contract-nli')

print('Data source import complete.')


Data source import complete.


## Data Engineering & Preparation

In [2]:
import os
import zipfile

# Your FIRST cell — just replace the extract_dir line:
extract_dir = seiftarek158_contract_nli_path
print(f"\nContents of {extract_dir}:")
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"{indent}  📄 {f}  ({size_kb:.1f} KB)")


Contents of /kaggle/input/datasets/seiftarek158/contract-nli:
📁 contract-nli/
  📁 contract-nli/
    📄 TERMS  (3.6 KB)
    📄 LICENSE  (18.2 KB)
    📄 dev.json  (1155.4 KB)
    📄 README.md  (5.4 KB)
    📄 train.json  (7429.9 KB)
    📄 test.json  (2207.2 KB)
    📁 raw/
      📄 1561604_0001193125-12-472390_d438799dex99d3.htm  (42.8 KB)
      📄 1277141_0001193125-15-184193_d923647dex99d2.htm  (41.8 KB)
      📄 clearcenter-rnda.pdf  (56.9 KB)
      📄 nda-format-approved-by-legal-section.pdf  (221.9 KB)
      📄 Non-Disclosure%20Agreement_3.pdf  (89.9 KB)
      📄 917253_0000917253-00-000008_document_8.txt  (23.4 KB)
      📄 Inaturals_NDA.pdf  (105.7 KB)
      📄 confagree.pdf  (125.1 KB)
      📄 828957_0000950109-98-001266_document_19.txt  (12.9 KB)
      📄 Non-disclosure%20Agreement_1.pdf  (164.8 KB)
      📄 59b1148ff6952b0001bdbedc_20170907_non%20disclosure%20agreement_expert.pdf  (175.7 KB)
      📄 NDA-Instructions-Agreement-Attachment.pdf  (123.6 KB)
      📄 25895_0000950134-07-023464_d513

In [3]:
import json

DATA_DIR = seiftarek158_contract_nli_path

with open(f"{DATA_DIR}/contract-nli/train.json") as f:
    train_data = json.load(f)

print("Top-level keys:", list(train_data.keys()))
print("Number of documents:", len(train_data["documents"]))
print("Number of hypotheses:", len(train_data["labels"]))
print("\nHypothesis IDs and texts:")
for h_id, h_text in train_data["labels"].items():
    print(f"  {h_id}: {h_text}")

Top-level keys: ['documents', 'labels']
Number of documents: 423
Number of hypotheses: 17

Hypothesis IDs and texts:
  nda-11: {'short_description': 'No reverse engineering', 'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}
  nda-16: {'short_description': 'Return of confidential information', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.'}
  nda-15: {'short_description': 'No licensing', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.'}
  nda-10: {'short_description': 'Confidentiality of Agreement', 'hypothesis': 'Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.'}
  nda-2: {'short_description': 'None-inclusion of non-technical information', 'hypothesis': 'Confidential Information shall only include technical information.'}
  nda-1: {'short_description'

In [4]:
doc = train_data["documents"][0]

print("Document ID:", doc["id"])
print("Contract length (chars):", len(doc["text"]))
print("Number of spans:", len(doc["spans"]))
print("\nFirst 3 spans (char ranges):")
for i, span in enumerate(doc["spans"][:3]):
    print(f"  span[{i}]: chars {span[0]}–{span[1]} → {repr(doc['text'][span[0]:span[1]][:80])}")

print("\nAnnotation keys:", list(doc["annotation_sets"][0]["annotations"].keys()))

print("\nSample annotation (nda-1):")
ann = doc["annotation_sets"][0]["annotations"]["nda-1"]
print("  choice:", ann["choice"])
print("  span indices:", ann["spans"])
if ann["spans"]:
    idx = ann["spans"][0]
    cs, ce = doc["spans"][idx]
    print(f"  resolved quote (span[{idx}], {cs}–{ce}): {repr(doc['text'][cs:ce][:120])}")

Document ID: 34
Contract length (chars): 8585
Number of spans: 65

First 3 spans (char ranges):
  span[0]: chars 0–44 → 'NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT'
  span[1]: chars 45–132 → 'This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”) is made by and b'
  span[2]: chars 133–331 → '(i) the Office of the United Nations High Commissioner for Refugees, having its '

Annotation keys: ['nda-11', 'nda-16', 'nda-15', 'nda-10', 'nda-2', 'nda-1', 'nda-19', 'nda-12', 'nda-20', 'nda-3', 'nda-18', 'nda-7', 'nda-17', 'nda-8', 'nda-13', 'nda-5', 'nda-4']

Sample annotation (nda-1):
  choice: Entailment
  span indices: [14]
  resolved quote (span[14], 1294–1683): '1. “Confidential Information”, whenever used in this Agreement, shall mean any data, document, specification and other i'


In [5]:
from collections import Counter, defaultdict

LABEL_MAP = {
    "Entailment": "ENTAILED",
    "Contradiction": "CONTRADICTED",
    "NotMentioned": "NOT_MENTIONED",
}

# Overall label distribution
overall_counter = Counter()
# Per-hypothesis distribution
hyp_counter = defaultdict(Counter)
# Evidence span count stats
evidence_counts = []
no_evidence_labels = Counter()  # labels that have spans=[]

for doc in train_data["documents"]:
    anns = doc["annotation_sets"][0]["annotations"]
    for h_id, ann in anns.items():
        label = LABEL_MAP[ann["choice"]]
        overall_counter[label] += 1
        hyp_counter[h_id][label] += 1
        evidence_counts.append(len(ann["spans"]))
        if len(ann["spans"]) == 0:
            no_evidence_labels[label] += 1

total = sum(overall_counter.values())
print("=== Overall Label Distribution ===")
for label, count in overall_counter.most_common():
    print(f"  {label:<20} {count:>5}  ({100*count/total:.1f}%)")

print(f"\n=== Evidence Span Coverage ===")
print(f"  Total (doc, hypothesis) pairs: {total}")
with_evidence = sum(1 for c in evidence_counts if c > 0)
print(f"  Pairs with ≥1 evidence span:   {with_evidence} ({100*with_evidence/total:.1f}%)")
print(f"  Pairs with 0 evidence spans:   {total - with_evidence} ({100*(total-with_evidence)/total:.1f}%)")
print(f"\n  Of zero-evidence pairs, by label:")
for label, count in no_evidence_labels.most_common():
    print(f"    {label:<20} {count}")

print(f"\n=== Per-Hypothesis Label Distribution ===")
for h_id in sorted(hyp_counter.keys(), key=lambda x: int(x.split('-')[1])):
    counts = hyp_counter[h_id]
    desc = train_data["labels"][h_id]["short_description"]
    print(f"  {h_id} ({desc})")
    for label in ["ENTAILED", "CONTRADICTED", "NOT_MENTIONED"]:
        c = counts.get(label, 0)
        print(f"    {label:<20} {c:>4}  ({100*c/sum(counts.values()):.1f}%)")

=== Overall Label Distribution ===
  ENTAILED              3530  (49.1%)
  NOT_MENTIONED         2820  (39.2%)
  CONTRADICTED           841  (11.7%)

=== Evidence Span Coverage ===
  Total (doc, hypothesis) pairs: 7191
  Pairs with ≥1 evidence span:   4371 (60.8%)
  Pairs with 0 evidence spans:   2820 (39.2%)

  Of zero-evidence pairs, by label:
    NOT_MENTIONED        2820

=== Per-Hypothesis Label Distribution ===
  nda-1 (Explicit identification)
    ENTAILED               91  (21.5%)
    CONTRADICTED          112  (26.5%)
    NOT_MENTIONED         220  (52.0%)
  nda-2 (None-inclusion of non-technical information)
    ENTAILED               23  (5.4%)
    CONTRADICTED          309  (73.0%)
    NOT_MENTIONED          91  (21.5%)
  nda-3 (Inclusion of verbally conveyed information)
    ENTAILED              270  (63.8%)
    CONTRADICTED            4  (0.9%)
    NOT_MENTIONED         149  (35.2%)
  nda-4 (Limited use)
    ENTAILED              364  (86.1%)
    CONTRADICTED            

In [6]:
import statistics

# Contract length distribution
lengths = [len(doc["text"]) for doc in train_data["documents"]]
print("=== Contract Length Stats (chars) ===")
print(f"  Min:    {min(lengths):>7}")
print(f"  Max:    {max(lengths):>7}")
print(f"  Mean:   {statistics.mean(lengths):>7.0f}")
print(f"  Median: {statistics.median(lengths):>7.0f}")
print(f"  Stdev:  {statistics.stdev(lengths):>7.0f}")

# Span count distribution
span_counts = [len(doc["spans"]) for doc in train_data["documents"]]
print(f"\n=== Spans per Document ===")
print(f"  Min:    {min(span_counts):>5}")
print(f"  Max:    {max(span_counts):>5}")
print(f"  Mean:   {statistics.mean(span_counts):>5.0f}")
print(f"  Median: {statistics.median(span_counts):>5.0f}")

# Load and sanity-check dev + test
with open(f"{DATA_DIR}/contract-nli/dev.json") as f:
    dev_data = json.load(f)
with open(f"{DATA_DIR}/contract-nli/test.json") as f:
    test_data = json.load(f)

print(f"\n=== Split Sizes ===")
print(f"  train: {len(train_data['documents'])} docs")
print(f"  dev:   {len(dev_data['documents'])} docs")
print(f"  test:  {len(test_data['documents'])} docs")
print(f"  total: {len(train_data['documents']) + len(dev_data['documents']) + len(test_data['documents'])} docs")

# Confirm test has no annotations (it's a held-out set)
test_doc = test_data["documents"][0]
first_ann = list(test_doc["annotation_sets"][0]["annotations"].values())[0]
print(f"\n=== Test Set Annotation Check ===")
print(f"  First annotation choice: {repr(first_ann['choice'])}")
print(f"  (Empty/null = blind test set, has labels = labeled test set)")

=== Contract Length Stats (chars) ===
  Min:       1481
  Max:      54571
  Mean:     11049
  Median:    9936
  Stdev:     6651

=== Spans per Document ===
  Min:       18
  Max:      354
  Mean:      78
  Median:    71

=== Split Sizes ===
  train: 423 docs
  dev:   61 docs
  test:  123 docs
  total: 607 docs

=== Test Set Annotation Check ===
  First annotation choice: 'NotMentioned'
  (Empty/null = blind test set, has labels = labeled test set)


In [7]:
import json

DATA_DIR = f"{DATA_DIR}/contract-nli"

LABEL_MAP = {
    "Entailment": "ENTAILED",
    "Contradiction": "CONTRADICTED",
    "NotMentioned": "NOT_MENTIONED",
}

def get_gold_label(doc, hypothesis_id):
    ann = doc["annotation_sets"][0]["annotations"].get(hypothesis_id)
    if ann is None:
        return None
    return LABEL_MAP.get(ann["choice"], ann["choice"])

def get_evidence_spans(doc, hypothesis_id):
    ann = doc["annotation_sets"][0]["annotations"].get(hypothesis_id)
    if ann is None:
        return []
    result = []
    for span_idx in ann.get("spans", []):
        char_start, char_end = doc["spans"][span_idx]
        result.append({
            "span_index": span_idx,
            "char_start": char_start,
            "char_end": char_end,
            "quote": doc["text"][char_start:char_end],
        })
    return result

def build_all_records(data):
    hypotheses = data["labels"]
    records = []
    for doc in data["documents"]:
        for h_id in hypotheses:
            label = get_gold_label(doc, h_id)
            if label is not None:
                records.append({
                    "doc_id": doc["id"],
                    "hypothesis_id": h_id,
                    "hypothesis_text": hypotheses[h_id]["hypothesis"],  # FIXED
                    "short_description": hypotheses[h_id]["short_description"],
                    "gold_label": label,
                    "evidence": get_evidence_spans(doc, h_id),
                    "contract_text": doc["text"],
                    "contract_length": len(doc["text"]),
                })
    return records

# Load all splits
with open(f"{DATA_DIR}/train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/dev.json") as f:
    dev_data = json.load(f)
with open(f"{DATA_DIR}/test.json") as f:
    test_data = json.load(f)

train_records = build_all_records(train_data)
dev_records   = build_all_records(dev_data)
test_records  = build_all_records(test_data)

print(f"Train records: {len(train_records)}")
print(f"Dev records:   {len(dev_records)}")
print(f"Test records:  {len(test_records)}")
print(f"Total:         {len(train_records) + len(dev_records) + len(test_records)}")
print(f"\nSample record keys: {list(train_records[0].keys())}")
print(f"Sample label: {train_records[0]['gold_label']}")
print(f"Sample hypothesis: {train_records[0]['hypothesis_text']}")
print(f"Sample evidence count: {len(train_records[0]['evidence'])}")

Train records: 7191
Dev records:   1037
Test records:  2091
Total:         10319

Sample record keys: ['doc_id', 'hypothesis_id', 'hypothesis_text', 'short_description', 'gold_label', 'evidence', 'contract_text', 'contract_length']
Sample label: NOT_MENTIONED
Sample hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
Sample evidence count: 0


In [8]:
# We'll estimate token counts without loading the actual model
# Rule of thumb: 1 token ≈ 4 chars for English legal text

def estimate_tokens(text):
    return len(text) // 4

# Analyze token budget for each record
hypothesis_avg_chars = sum(len(r["hypothesis_text"]) for r in train_records) / len(train_records)
print(f"=== Token Budget Analysis ===")
print(f"Avg hypothesis length (chars): {hypothesis_avg_chars:.0f} (~{hypothesis_avg_chars//4:.0f} tokens)")

# Prompt overhead (template text itself, not counting contract/hypothesis)
PROMPT_TEMPLATE_OVERHEAD = 60  # chars for "Contract:\n...\nHypothesis:\n...\nLabel:" etc.
print(f"Prompt template overhead: ~{PROMPT_TEMPLATE_OVERHEAD} chars (~{PROMPT_TEMPLATE_OVERHEAD//4} tokens)")

# Contract token distribution
contract_token_estimates = [estimate_tokens(r["contract_text"]) for r in train_records]
# deduplicate by doc_id since contract_text repeats across hypotheses
seen = set()
unique_contract_tokens = []
for r in train_records:
    if r["doc_id"] not in seen:
        seen.add(r["doc_id"])
        unique_contract_tokens.append(estimate_tokens(r["contract_text"]))

import statistics
print(f"\n=== Unique Contract Token Estimates (train, {len(unique_contract_tokens)} docs) ===")
print(f"  Min:    {min(unique_contract_tokens):>6}")
print(f"  Max:    {max(unique_contract_tokens):>6}")
print(f"  Mean:   {statistics.mean(unique_contract_tokens):>6.0f}")
print(f"  Median: {statistics.median(unique_contract_tokens):>6.0f}")
print(f"  Stdev:  {statistics.stdev(unique_contract_tokens):>6.0f}")

# How many contracts fit in common context windows?
for limit in [1024, 2048, 4096, 8192]:
    fits = sum(1 for t in unique_contract_tokens if t <= limit)
    pct = 100 * fits / len(unique_contract_tokens)
    print(f"  Contracts fitting in {limit:>5} tokens: {fits:>4} / {len(unique_contract_tokens)} ({pct:.1f}%)")

# Evidence span token lengths
evidence_token_lengths = []
for r in train_records:
    for e in r["evidence"]:
        evidence_token_lengths.append(estimate_tokens(e["quote"]))

print(f"\n=== Evidence Span Token Lengths ===")
print(f"  Count:  {len(evidence_token_lengths)}")
print(f"  Min:    {min(evidence_token_lengths)}")
print(f"  Max:    {max(evidence_token_lengths)}")
print(f"  Mean:   {statistics.mean(evidence_token_lengths):.0f}")
print(f"  Median: {statistics.median(evidence_token_lengths):.0f}")

=== Token Budget Analysis ===
Avg hypothesis length (chars): 97 (~24 tokens)
Prompt template overhead: ~60 chars (~15 tokens)

=== Unique Contract Token Estimates (train, 423 docs) ===
  Min:       370
  Max:     13642
  Mean:     2762
  Median:   2484
  Stdev:    1663
  Contracts fitting in  1024 tokens:   40 / 423 (9.5%)
  Contracts fitting in  2048 tokens:  159 / 423 (37.6%)
  Contracts fitting in  4096 tokens:  354 / 423 (83.7%)
  Contracts fitting in  8192 tokens:  419 / 423 (99.1%)

=== Evidence Span Token Lengths ===
  Count:  8341
  Min:    0
  Max:    401
  Mean:   63
  Median: 50


In [9]:
import json, statistics

# Compute relative position of each evidence span within its contract
positions = []  # (relative_position 0-1, hypothesis_id, label)

for doc in train_data["documents"]:
    contract_len = len(doc["text"])
    anns = doc["annotation_sets"][0]["annotations"]
    for h_id, ann in anns.items():
        label = LABEL_MAP[ann["choice"]]
        for span_idx in ann.get("spans", []):
            cs, ce = doc["spans"][span_idx]
            mid = (cs + ce) / 2
            rel = mid / contract_len  # 0 = start, 1 = end
            positions.append({
                "rel_pos": rel,
                "h_id": h_id,
                "label": label,
                "char_start": cs,
                "char_end": ce,
                "contract_len": contract_len
            })

# Bucket into 10 bins (deciles)
buckets = [0] * 10
for p in positions:
    bucket = min(int(p["rel_pos"] * 10), 9)
    buckets[bucket] += 1

total_spans = len(positions)
bucket_pcts = [round(100 * b / total_spans, 1) for b in buckets]

print("Decile distribution of evidence spans:")
for i, pct in enumerate(bucket_pcts):
    bar = "█" * int(pct)
    print(f"  {i*10:>2}-{(i+1)*10}%  {bar} {pct}%")

# Cumulative coverage
cumulative = 0
print("\nCumulative coverage by truncation point:")
for i, pct in enumerate(bucket_pcts):
    cumulative += pct
    print(f"  Keep first {(i+1)*10}% of contract → covers {cumulative:.1f}% of evidence spans")

# Save for visualization
print("\nbucket_pcts =", bucket_pcts)

Decile distribution of evidence spans:
   0-10%  █████ 5.6%
  10-20%  ██████████████████ 18.2%
  20-30%  █████████████████ 17.9%
  30-40%  ████████████████ 16.3%
  40-50%  █████████████ 13.5%
  50-60%  ███████████ 11.6%
  60-70%  ███████ 7.8%
  70-80%  █████ 5.1%
  80-90%  ██ 2.3%
  90-100%  █ 1.7%

Cumulative coverage by truncation point:
  Keep first 10% of contract → covers 5.6% of evidence spans
  Keep first 20% of contract → covers 23.8% of evidence spans
  Keep first 30% of contract → covers 41.7% of evidence spans
  Keep first 40% of contract → covers 58.0% of evidence spans
  Keep first 50% of contract → covers 71.5% of evidence spans
  Keep first 60% of contract → covers 83.1% of evidence spans
  Keep first 70% of contract → covers 90.9% of evidence spans
  Keep first 80% of contract → covers 96.0% of evidence spans
  Keep first 90% of contract → covers 98.3% of evidence spans
  Keep first 100% of contract → covers 100.0% of evidence spans

bucket_pcts = [5.6, 18.2, 17.9, 16.3

In [10]:
import json

# Truncation limit for contract text (in chars, since we use 4 chars ≈ 1 token)
# CONTRACT_CHAR_LIMIT = 3800 * 4  # = 15200 chars

# max_length=2048
CONTRACT_TOKEN_LIMIT = 1700
CONTRACT_CHAR_LIMIT = CONTRACT_TOKEN_LIMIT * 4
# CONTRACT_CHAR_LIMIT = 1700 * 4   # = 3400 chars
# max_length=512
# CONTRACT_CHAR_LIMIT = 340 * 4   # = 1360 chars

def build_prompt(record, include_answer=True):
    """
    Build the full prompt for a single (doc, hypothesis) pair.
    include_answer=True  → training format (input + output)
    include_answer=False → inference format (input only)

    Truncation strategy: skip first 10% of contract (only 5.6% of evidence spans),
    then take CONTRACT_CHAR_LIMIT chars. Covers ~65.9% of evidence spans at max_length=1024.
    Evidence spans outside the visible window are dropped from the training target.
    """
    contract = record["contract_text"]

    if include_answer:
        CONTRACT_TOKEN_LIMIT = 1400  # leaves ~400 tokens for output
    else:
        CONTRACT_TOKEN_LIMIT = 1700  # 7200 chars — no output needed


    CONTRACT_CHAR_LIMIT = CONTRACT_TOKEN_LIMIT * 4
    
    if len(contract) > CONTRACT_CHAR_LIMIT:
        offset = int(len(contract) * 0.10)
        visible_start = offset
        visible_end = offset + CONTRACT_CHAR_LIMIT
        contract = contract[offset : offset + CONTRACT_CHAR_LIMIT] + "\n[TRUNCATED]"
    else:
        visible_start = 0
        visible_end = len(contract)

    input_part = (
        f"Classify the hypothesis based on the contract. "
        f"Respond with ONLY valid JSON nothing else:\n"
        f'{{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "exact quote 2"]}}\n'
        f"If label is NOT_MENTIONED, evidence must be [].\n"
        f"Evidence must be copied verbatim from the contract text, word for word. Do not paraphrase or invent.\n"
        f"Contract:\n{contract}\n\n"
        f"Hypothesis:\n{record['hypothesis_text']}\n\n"
    )

    if not include_answer:
        return input_part

    # Only include evidence quotes visible in the truncated window
    visible_evidence = [
        e["quote"].strip().replace("\n", " ")
        for e in record["evidence"]
        if e["char_start"] >= visible_start and e["char_end"] <= visible_end
    ]

    output_part = json.dumps({
        "label": record["gold_label"],
        "evidence": visible_evidence
    })

    return input_part + output_part


# Test on a few samples
for label_target in ["ENTAILED", "CONTRADICTED", "NOT_MENTIONED"]:
    sample = next(r for r in train_records if r["gold_label"] == label_target)
    prompt = build_prompt(sample, include_answer=True)
    tokens_est = len(prompt) // 4
    print(f"=== {label_target} example ===")
    print(f"  Total prompt length: {len(prompt)} chars (~{tokens_est} tokens)")
    print(f"  Contract truncated: {len(sample['contract_text']) > CONTRACT_CHAR_LIMIT}")
    print(f"  --- PROMPT (trimmed to 300 chars) ---")
    print(prompt[:300])
    print(f"  --- TAIL (last 200 chars) ---")
    print(prompt[-200:])
    print()

=== ENTAILED example ===
  Total prompt length: 6453 chars (~1613 tokens)
  Contract truncated: True
  --- PROMPT (trimmed to 300 chars) ---
Classify the hypothesis based on the contract. Respond with ONLY valid JSON nothing else:
{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "exact quote 2"]}
If label is NOT_MENTIONED, evidence must be [].
Evidence must be copied verbatim from the contract text, 
  --- TAIL (last 200 chars) ---
ned to UNHCR or destroyed:", "(a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or"]}

=== CONTRADICTED example ===
  Total prompt length: 6089 chars (~1522 tokens)
  Contract truncated: True
  --- PROMPT (trimmed to 300 chars) ---
Classify the hypothesis based on the contract. Respond with ONLY valid JSON nothing else:
{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "e

In [11]:
import json
import os

OUTPUT_DIR = "/kaggle/working/preprocessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def serialize_split(records, split_name):
    output = []
    truncated_count = 0

    for r in records:
        was_truncated = len(r["contract_text"]) > CONTRACT_CHAR_LIMIT
        if was_truncated:
            truncated_count += 1

        output.append({
            # Identifiers
            "doc_id": r["doc_id"],
            "hypothesis_id": r["hypothesis_id"],
            "split": split_name,

            # Labels & evidence
            "gold_label": r["gold_label"],
            "evidence": r["evidence"],

            # Prompts
            "prompt_train": build_prompt(r, include_answer=True),
            "prompt_inference": build_prompt(r, include_answer=False),

            # Metadata
            "contract_length_chars": r["contract_length"],
            "was_truncated": was_truncated,
            "hypothesis_text": r["hypothesis_text"],
            "short_description": r["short_description"],
        })

    path = f"{OUTPUT_DIR}/{split_name}.json"
    with open(path, "w") as f:
        json.dump(output, f, indent=2)

    print(f"=== {split_name} ===")
    print(f"  Records written:  {len(output)}")
    print(f"  Truncated:        {truncated_count} ({100*truncated_count/len(output):.1f}%)")
    print(f"  Saved to:         {path}")
    print(f"  File size:        {os.path.getsize(path)/1024/1024:.1f} MB")

    return output

train_out = serialize_split(train_records, "train")
dev_out   = serialize_split(dev_records,   "dev")
test_out  = serialize_split(test_records,  "test")

# Sanity check — verify a random record round-trips correctly
import random
sample = random.choice(train_out)
print(f"\n=== Round-trip Sanity Check ===")
print(f"  doc_id:         {sample['doc_id']}")
print(f"  hypothesis_id:  {sample['hypothesis_id']}")
print(f"  gold_label:     {sample['gold_label']}")
print(f"  was_truncated:  {sample['was_truncated']}")
print(f"  train prompt ends with: {repr(sample['prompt_train'][-60:])}")
print(f"  inference prompt ends with: {repr(sample['prompt_inference'][-60:])}")

=== train ===
  Records written:  7191
  Truncated:        5066 (70.4%)
  Saved to:         /kaggle/working/preprocessed/train.json
  File size:        94.6 MB
=== dev ===
  Records written:  1037
  Truncated:        816 (78.7%)
  Saved to:         /kaggle/working/preprocessed/dev.json
  File size:        14.0 MB
=== test ===
  Records written:  2091
  Truncated:        1479 (70.7%)
  Saved to:         /kaggle/working/preprocessed/test.json
  File size:        27.3 MB

=== Round-trip Sanity Check ===
  doc_id:         123
  hypothesis_id:  nda-12
  gold_label:     NOT_MENTIONED
  was_truncated:  False
  train prompt ends with: 'ial Information.\n\n{"label": "NOT_MENTIONED", "evidence": []}'
  inference prompt ends with: 'y develop information similar to Confidential Information.\n\n'


## Base Model Selection & QLora Setup

In [12]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # it might be tempting to remove this LINE, but I beg you dont, I suffered before because I did

In [13]:
from getpass import getpass
HF_API_TOKEN = getpass("Enter your HF token: ")

from huggingface_hub import login
login(token=HF_API_TOKEN)


In [14]:
!pip install --upgrade unsloth unsloth_zoo

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.4
    Uninstalling datasets-4.8.4:
      Successfully uninstalled datasets-4.8.4
  Attempting uninstall: trl
    Found existing installation: trl 1.0.0
    Uninstalling trl-1.0.0:
      Successfully uninstalled trl-1.0.0


In [15]:
! pip install trl==1.0.0
! pip install -U "bitsandbytes>=0.46.1"

  Using cached trl-1.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
Using cached trl-1.0.0-py3-none-any.whl (630 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.
unsloth-zoo 2026.4.2 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.0.0 which is incompatible.
unsloth 2026.4.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, bu

In [16]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [17]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
from datasets import load_dataset, Dataset
from peft import (
prepare_model_for_kbit_training,
LoraConfig,
get_peft_model,
AutoPeftModelForCausalLM,
)
from trl import SFTConfig, SFTTrainer

In [18]:
quant_config = BitsAndBytesConfig(
load_in_4bit=True,
bnb_4bit_use_double_quant=True,
bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.float16,
)

In [19]:
base_model_name = 'Qwen/Qwen3-4B-Instruct-2507'

In [20]:


model, tokenizer = FastLanguageModel.from_pretrained(
      base_model_name,
      max_seq_length=2048,
      load_in_4bit=True,
  )
 

==((====))==  Unsloth 2026.4.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [21]:
record = test_records[0]

prompt = build_prompt(record, include_answer=False)
print(f"=== Sample Prompt ===")
print(prompt[:200])

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
    )

=== Sample Prompt ===
Classify the hypothesis based on the contract. Respond with ONLY valid JSON nothing else:
{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "exact quote 2"]}
If la


Both `max_new_tokens` (=150) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

In [22]:
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Hypothesis:   {record['hypothesis_text']}")
print(f"Gold label:   {record['gold_label']}")
print(f"Prediction:   {generated.strip()}")

Hypothesis:   Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
Gold label:   NOT_MENTIONED
Prediction:   Classification:
"NOT_MENTIONED"
Evidence: []


In [23]:
train_ds = Dataset.from_list([{"text": r["prompt_train"]} for r in train_out])
dev_ds   = Dataset.from_list([{"text": r["prompt_train"]} for r in dev_out])

# 2. Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# 3. LoRA config
model = FastLanguageModel.get_peft_model(
      model, r=12, lora_alpha=32, lora_dropout=0.15,
      target_modules=["q_proj","k_proj","v_proj","o_proj"],
  )
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.15.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.1 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 8,847,360 || all params: 4,031,315,456 || trainable%: 0.2195


In [24]:
training_arguments = SFTConfig(
fp16=False,
bf16=False,
dataset_text_field="text",
per_device_train_batch_size=2,
gradient_accumulation_steps=8,
gradient_checkpointing=True,
optim="paged_adamw_8bit",
learning_rate=1e-4,
num_train_epochs=2,
eval_strategy="steps",
eval_steps=0.2, # evaluate every 20% of an epoch
save_strategy="steps",
output_dir="./epoch-finetuned-qwen3-4b",
logging_steps=20,
report_to="none",
seed=42,
max_length=2048,
packing=False,
padding_free=False,
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [25]:
trainer = SFTTrainer(
model=model,
train_dataset=train_ds,
eval_dataset=dev_ds,
args=training_arguments,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/7191 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1037 [00:00<?, ? examples/s]

In [26]:
trainer.train(resume_from_checkpoint="./epoch-finetuned-qwen3-4b/checkpoint-500")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151654}.
	eval_steps: 0.2 (from args) != 180 (from trainer_state.json)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,191 | Num Epochs = 2 | Total steps = 900
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,847,360 of 4,031,315,456 (0.22% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
540,0.976838,1.477960
720,0.882596,1.555324
900,0.861985,1.586661


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=900, training_loss=0.399566338857015, metrics={'train_runtime': 15272.4802, 'train_samples_per_second': 0.942, 'train_steps_per_second': 0.059, 'total_flos': 3.911253557139763e+17, 'train_loss': 0.399566338857015})

In [27]:
model.save_pretrained("./qwen3-4B-nli-lora-adapter")
tokenizer.save_pretrained("./qwen3-4B-nli-lora-adapter")

('./qwen3-4B-nli-lora-adapter/tokenizer_config.json',
 './qwen3-4B-nli-lora-adapter/chat_template.jinja',
 './qwen3-4B-nli-lora-adapter/tokenizer.json')

In [31]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

ADAPTER_PATH = "/kaggle/working/qwen3-4B-nli-lora-adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(model, "./qwen3-4B-nli-lora-adapter")
model.eval()

print("Model loaded!")

==((====))==  Unsloth 2026.4.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded!


In [45]:
record = test_records[1000]

prompt = build_prompt(record, include_answer=False)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=None,    # required when do_sample=False for Qwen3
        top_p=None,          # same
    )

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:], 
    skip_special_tokens=True
)

print(f"Hypothesis:   {record['hypothesis_text']}")
print(f"Gold label:   {record['gold_label']}")
print(f"Prediction: {generated}")

Both `max_new_tokens` (=150) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hypothesis:   Receiving Party may acquire information similar to Confidential Information from a third party.
Gold label:   ENTAILED
Prediction: {"label": "ENTAILED", "evidence": ["The foregoing shall not prevent either party from disclosing Information which is:", "iii) rightfully received from a third party; or"]}


In [33]:
!zip -r /kaggle/working/qwen-nli-lora-adapter.zip /kaggle/working/qwen3-4B-nli-lora-adapter

  adding: kaggle/working/qwen3-4B-nli-lora-adapter/ (stored 0%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/adapter_config.json (deflated 56%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/tokenizer.json (deflated 81%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/README.md (deflated 65%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/chat_template.jinja (deflated 76%)
  adding: kaggle/working/qwen3-4B-nli-lora-adapter/tokenizer_config.json (deflated 43%)
